# LeetCode 3917 — Count Indices With Opposite Parity bằng Python

Link đề: https://leetcode.com/problems/count-indices-with-opposite-parity/ · Độ khó: **Easy** · Chủ đề: Array, Parity, Suffix Counting

Skeleton LeetCode:

```python
class Solution:
    def countOppositeParity(self, nums: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Weekly Contest 500, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi chỉ số `i`, đếm số chỉ số `j > i` mà `nums[i]` và `nums[j]` có **tính chẵn/lẻ khác nhau**.
2. Biết tính **parity** (`x % 2`), và đếm theo hai nhóm (chẵn / lẻ) thay vì so sánh từng cặp.
3. Nắm kỹ thuật **đếm hậu tố (suffix count)** duyệt từ phải sang trái, và kỹ thuật **giảm dần từ tổng** duyệt từ trái sang phải.
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: đếm cả `j < i`, đếm cùng parity thay vì khác parity, tính cả chính `i`, quên cập nhật bộ đếm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n`. **Điểm** của chỉ số `i` là số chỉ số `j` thỏa `i < j < n` và `nums[i]`, `nums[j]` có **tính chẵn lẻ khác nhau** (một chẵn một lẻ). Trả về mảng `answer` độ dài `n` với `answer[i]` là điểm của `i`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3,4]` | `[2,1,1,0]` | `1` (lẻ): sau nó có `2` và `4` (chẵn) -> 2; `2` (chẵn): sau nó có `3` (lẻ) -> 1; `3` (lẻ): sau nó có `4` (chẵn) -> 1; `4`: không có số lẻ phía sau -> 0 |
| 2 | `[1]` | `[0]` | Không có chỉ số `j` nào lớn hơn |

Constraints: `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
nums = [1, 2, 3, 4]
for i in range(len(nums)):
    after = nums[i + 1:]
    diff = [y for y in after if y % 2 != nums[i] % 2]
    print(f"i={i} nums[i]={nums[i]} phía sau={after} khác parity={diff} -> {len(diff)}")

i=0 nums[i]=1 phía sau=[2, 3, 4] khác parity=[2, 4] -> 2
i=1 nums[i]=2 phía sau=[3, 4] khác parity=[3] -> 1
i=2 nums[i]=3 phía sau=[4] khác parity=[4] -> 1
i=3 nums[i]=4 phía sau=[] khác parity=[] -> 0


## 3. Kiến thức nền

### 3.1. Chẵn/lẻ (parity)

`x % 2 == 0` nếu `x` chẵn, `x % 2 == 1` nếu `x` lẻ. Hai số có parity **khác nhau** khi `a % 2 != b % 2`, hoặc dùng XOR bit cuối `(a ^ b) & 1`.

In [2]:
print([x % 2 for x in [1, 2, 3, 4]])
print(3 % 2 != 4 % 2, (3 ^ 4) & 1, (3 ^ 5) & 1)

[1, 0, 1, 0]
True 1 0


### 3.2. Hai bộ đếm theo nhóm

Thay vì so từng cặp, ta đếm **mỗi nhóm** có bao nhiêu phần tử: `even` và `odd`. Số phần tử khác parity với một số lẻ chính là `even`, với một số chẵn là `odd`. Có thể gói hai bộ đếm vào một list `cnt = [số_chẵn, số_lẻ]` và truy cập `cnt[x % 2]` (cùng parity) hoặc `cnt[1 - x % 2]` (khác parity).

In [3]:
cnt = [0, 0]                 # cnt[0] = số chẵn, cnt[1] = số lẻ
for x in [1, 2, 3, 4]:
    cnt[x % 2] += 1
print(cnt)
x = 3
print("khác parity với", x, ":", cnt[1 - x % 2])

[2, 2]
khác parity với 3 : 2


### 3.3. Duyệt ngược `range(n - 1, -1, -1)`

Ta duyệt từ phải sang trái để các phần tử **phía sau** `i` đã được đếm khi tới `i`. `reversed(range(n))` cũng được.

In [4]:
n = 4
print(list(range(n - 1, -1, -1)), list(reversed(range(n))))

[3, 2, 1, 0] [3, 2, 1, 0]


### 3.4. `itertools.accumulate`

`accumulate(iterable)` cho tổng tích lũy. Với dãy bit parity đảo ngược ta nhận được số số lẻ trong từng hậu tố.

In [5]:
from itertools import accumulate
bits = [x % 2 for x in [1, 2, 3, 4]]
print(bits, list(accumulate(reversed(bits)))[::-1])      # số số lẻ trong nums[i:]

[1, 0, 1, 0] [2, 1, 1, 0]


### 3.5. Thứ tự "đọc rồi mới cập nhật"

Khi duyệt ngược, tại `i` ta **đọc** bộ đếm (chỉ chứa các phần tử phía sau `i`) rồi **mới** thêm `nums[i]` vào bộ đếm. Đảo thứ tự hai bước sẽ đếm cả chính `i`.

In [6]:
cnt = [0, 0]
ans = []
for x in reversed([1, 2, 3, 4]):
    ans.append(cnt[1 - x % 2])          # đọc: chỉ phần tử phía sau
    cnt[x % 2] += 1                     # cập nhật: thêm x
print(ans[::-1])

[2, 1, 1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với mỗi `i` ta chỉ quan tâm **có bao nhiêu số chẵn và bao nhiêu số lẻ ở phía sau `i`**. Nếu `nums[i]` lẻ, điểm = số **chẵn** phía sau; nếu `nums[i]` chẵn, điểm = số **lẻ** phía sau.

Tính tay `[1, 2, 3, 4]` duyệt từ phải sang trái, giữ `(even, odd)` của các phần tử đã đi qua:

| i | nums[i] | parity | even/odd phía sau | điểm | sau khi thêm nums[i] |
|---|---|---|---|---|---|
| 3 | 4 | chẵn | (0, 0) | 0 (lẻ phía sau: 0) | (1, 0) |
| 2 | 3 | lẻ | (1, 0) | 1 (chẵn phía sau: 1) | (1, 1) |
| 1 | 2 | chẵn | (1, 1) | 1 (lẻ phía sau: 1) | (2, 1) |
| 0 | 1 | lẻ | (2, 1) | 2 (chẵn phía sau: 2) | (2, 2) |

Đáp án `[2, 1, 1, 0]` (đọc từ `i = 0`).

In [7]:
def explain(nums):
    even = odd = 0
    ans = [0] * len(nums)
    for i in range(len(nums) - 1, -1, -1):
        ans[i] = even if nums[i] % 2 == 1 else odd
        print(f"i={i} x={nums[i]} (phía sau: chẵn={even}, lẻ={odd}) -> {ans[i]}")
        if nums[i] % 2 == 0:
            even += 1
        else:
            odd += 1
    return ans

print(explain([1, 2, 3, 4]))

i=3 x=4 (phía sau: chẵn=0, lẻ=0) -> 0
i=2 x=3 (phía sau: chẵn=1, lẻ=0) -> 1
i=1 x=2 (phía sau: chẵn=1, lẻ=1) -> 1
i=0 x=1 (phía sau: chẵn=2, lẻ=1) -> 2
[2, 1, 1, 0]


## 5. Cách 1 — Brute Force / baseline

Hai vòng lặp: với mỗi `i`, duyệt mọi `j > i`, đếm các `j` có `nums[i] % 2 != nums[j] % 2`. Đúng định nghĩa của đề.

Nút thắt: tổng `O(n²)` phép so sánh; mỗi `i` đếm lại các phần tử phía sau mà `i + 1` cũng sẽ đếm. Chỉ cần nhớ số chẵn/lẻ của hậu tố.

In [8]:
class SolutionBrute:
    def countOppositeParity(self, nums):
        n = len(nums)
        ans = []
        for i in range(n):
            count = 0
            for j in range(i + 1, n):
                if nums[i] % 2 != nums[j] % 2:
                    count += 1
            ans.append(count)
        return ans

print(SolutionBrute().countOppositeParity([1, 2, 3, 4]), SolutionBrute().countOppositeParity([1]))

[2, 1, 1, 0] [0]


**Độ phức tạp:** Time `O(n²)`, Space `O(n)` cho kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** nhóm theo parity. Điểm của `i` chỉ phụ thuộc hai số: số chẵn và số lẻ **phía sau `i`**. Hai số này **tăng dần** khi `i` giảm (thêm một phần tử mỗi bước), nên duy trì được bằng hai biến.

**Invariant** (duyệt `i` từ `n-1` về `0`): trước khi xử lý `i`, `even` và `odd` bằng số phần tử chẵn/lẻ trong `nums[i+1:]`. Ta đọc điểm của `i` từ biến thích hợp, rồi cập nhật biến theo parity của `nums[i]`.

Cách đối ngẫu: duyệt từ **trái** với tổng ban đầu `(even_total, odd_total)`, **bỏ** `nums[i]` khỏi nhóm của nó rồi đọc điểm (lúc này bộ đếm còn `nums[i+1:]`).

Cũng có công thức: nếu `suf_odd[i]` là số lẻ trong `nums[i+1:]` thì điểm là `suf_odd[i]` khi `nums[i]` chẵn và `(n - 1 - i) - suf_odd[i]` khi lẻ.

## 7. Các cách tiếp theo

### Cách 2 — Đếm hậu tố từ phải sang trái

Dùng `even`, `odd` như mục 6.

In [9]:
class SolutionSuffix:
    def countOppositeParity(self, nums):
        n = len(nums)
        ans = [0] * n
        even = odd = 0
        for i in range(n - 1, -1, -1):
            ans[i] = even if nums[i] % 2 == 1 else odd
            if nums[i] % 2 == 0:
                even += 1
            else:
                odd += 1
        return ans

print(SolutionSuffix().countOppositeParity([1, 2, 3, 4]), SolutionSuffix().countOppositeParity([1]))

[2, 1, 1, 0] [0]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (kết quả), thêm `O(1)`.

### Cách 3 — Từ tổng toàn mảng, bỏ dần từ trái

Tính `cnt = [even_total, odd_total]`. Với `i` từ trái: bỏ `nums[i]` khỏi nhóm của nó, điểm là phần tử còn lại **ở nhóm khác parity**.

In [10]:
class SolutionTotals:
    def countOppositeParity(self, nums):
        cnt = [0, 0]
        for x in nums:
            cnt[x % 2] += 1
        ans = []
        for x in nums:
            cnt[x % 2] -= 1                  # bỏ x: còn lại đúng nums[i+1:]
            ans.append(cnt[1 - x % 2])
        return ans

print(SolutionTotals().countOppositeParity([1, 2, 3, 4]), SolutionTotals().countOppositeParity([1]))

[2, 1, 1, 0] [0]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (kết quả).

### Cách 4 — Mảng số lẻ hậu tố rồi công thức

Dựng `suf_odd[i]` = số lẻ trong `nums[i+1:]` (một mảng độ dài `n + 1`). Rồi `ans[i] = suf_odd[i]` nếu `nums[i]` chẵn, ngược lại `(n - 1 - i) - suf_odd[i]`.

In [11]:
class SolutionSufOdd:
    def countOppositeParity(self, nums):
        n = len(nums)
        suf_odd = [0] * (n + 1)              # suf_odd[i] = số lẻ trong nums[i:]
        for i in range(n - 1, -1, -1):
            suf_odd[i] = suf_odd[i + 1] + nums[i] % 2
        ans = []
        for i, x in enumerate(nums):
            odd_after = suf_odd[i + 1]
            even_after = (n - 1 - i) - odd_after
            ans.append(even_after if x % 2 == 1 else odd_after)
        return ans

print(SolutionSufOdd().countOppositeParity([1, 2, 3, 4]), SolutionSufOdd().countOppositeParity([1]))

[2, 1, 1, 0] [0]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 5 — `accumulate` trên dãy bit parity

Dùng `accumulate` để tính số lẻ hậu tố rồi ghép; ngắn gọn bằng một vài dòng.

In [12]:
from itertools import accumulate

class SolutionAccumulate:
    def countOppositeParity(self, nums):
        n = len(nums)
        bits = [x % 2 for x in nums]
        odd_suffix = list(accumulate(reversed(bits)))[::-1] + [0]   # odd_suffix[i] = số lẻ trong nums[i:]
        res = []
        for i, b in enumerate(bits):
            odd_after = odd_suffix[i + 1]
            res.append((n - 1 - i - odd_after) if b else odd_after)
        return res

print(SolutionAccumulate().countOppositeParity([1, 2, 3, 4]), SolutionAccumulate().countOppositeParity([1]))

[2, 1, 1, 0] [0]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 (bỏ dần từ trái) với `nums = [1, 2, 3, 4]`: `cnt = [2, 2]` (2 chẵn, 2 lẻ).

| i | x | parity | `cnt` trước | sau khi bỏ x | điểm = `cnt[1 - parity]` |
|---|---|---|---|---|---|
| 0 | 1 | lẻ (1) | [2, 2] | [2, 1] | cnt[0] = 2 |
| 1 | 2 | chẵn (0) | [2, 1] | [1, 1] | cnt[1] = 1 |
| 2 | 3 | lẻ (1) | [1, 1] | [1, 0] | cnt[0] = 1 |
| 3 | 4 | chẵn (0) | [1, 0] | [0, 0] | cnt[1] = 0 |

In [13]:
def trace_totals(nums):
    cnt = [0, 0]
    for x in nums:
        cnt[x % 2] += 1
    print("tổng: chẵn =", cnt[0], "lẻ =", cnt[1])
    ans = []
    for i, x in enumerate(nums):
        before = cnt[:]
        cnt[x % 2] -= 1
        ans.append(cnt[1 - x % 2])
        print(f"i={i} x={x} parity={x % 2}: cnt {before} -> {cnt}; điểm = cnt[{1 - x % 2}] = {ans[-1]}")
    return ans

assert trace_totals([1, 2, 3, 4]) == [2, 1, 1, 0]
assert trace_totals([1]) == [0]

tổng: chẵn = 2 lẻ = 2
i=0 x=1 parity=1: cnt [2, 2] -> [2, 1]; điểm = cnt[0] = 2
i=1 x=2 parity=0: cnt [2, 1] -> [1, 1]; điểm = cnt[1] = 1
i=2 x=3 parity=1: cnt [1, 1] -> [1, 0]; điểm = cnt[0] = 1
i=3 x=4 parity=0: cnt [1, 0] -> [0, 0]; điểm = cnt[1] = 0
tổng: chẵn = 0 lẻ = 1
i=0 x=1 parity=1: cnt [0, 1] -> [0, 0]; điểm = cnt[0] = 0


In [14]:
def trace_suffix(nums):
    even = odd = 0
    n = len(nums)
    ans = [0] * n
    for i in range(n - 1, -1, -1):
        ans[i] = even if nums[i] % 2 else odd
        print(f"i={i} x={nums[i]} phía sau (chẵn={even}, lẻ={odd}) -> ans[{i}]={ans[i]}")
        if nums[i] % 2:
            odd += 1
        else:
            even += 1
    return ans

assert trace_suffix([2, 4, 6]) == [0, 0, 0]
assert trace_suffix([1, 3, 2]) == [1, 1, 0]

i=2 x=6 phía sau (chẵn=0, lẻ=0) -> ans[2]=0
i=1 x=4 phía sau (chẵn=1, lẻ=0) -> ans[1]=0
i=0 x=2 phía sau (chẵn=2, lẻ=0) -> ans[0]=0
i=2 x=2 phía sau (chẵn=0, lẻ=0) -> ans[2]=0
i=1 x=3 phía sau (chẵn=1, lẻ=0) -> ans[1]=1
i=0 x=1 phía sau (chẵn=1, lẻ=1) -> ans[0]=1


## 9. Code nộp LeetCode

Dùng Cách 2 (đếm hậu tố, duyệt từ phải sang trái).

In [15]:
from typing import List, Optional


class Solution:
    def countOppositeParity(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [0] * n
        even = odd = 0                      # số chẵn / lẻ trong nums[i+1:]
        for i in range(n - 1, -1, -1):
            ans[i] = even if nums[i] % 2 == 1 else odd    # lẻ -> đếm chẵn; chẵn -> đếm lẻ
            if nums[i] % 2 == 0:
                even += 1
            else:
                odd += 1
        return ans


print(Solution().countOppositeParity([1, 2, 3, 4]), Solution().countOppositeParity([1]))

[2, 1, 1, 0] [0]


## 10. Vì sao thuật toán đúng

1. **Parity khác nhau chỉ có hai khả năng.** `nums[i]` lẻ thì chỉ các `nums[j]` chẵn khác parity; `nums[i]` chẵn thì chỉ các `nums[j]` lẻ.
2. **Invariant.** Trước khi xử lý `i`, `even = #{j > i : nums[j] chẵn}` và `odd = #{j > i : nums[j] lẻ}`. Ban đầu (`i = n - 1`) đều `0` đúng.
3. **Đọc trước khi cập nhật.** `ans[i]` lấy từ `even` hoặc `odd` chưa tính `nums[i]`, nên chỉ đếm `j > i`.
4. **Cập nhật giữ invariant.** Sau khi thêm `nums[i]` vào nhóm của nó, `even`/`odd` bằng số phần tử chẵn/lẻ trong `nums[i..]` = `nums[(i-1)+1:]` — invariant đúng cho `i - 1`.
5. **Phép đối ngẫu** (Cách 3): bỏ `nums[i]` khỏi tổng cho ra số phần tử của `nums[i+1:]` — cùng giá trị.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- Cách 1: `O(n²)` thời gian, `O(n)` bộ nhớ cho kết quả.
- Cách 2, 3: `O(n)` thời gian; kết quả `O(n)`, bộ nhớ phụ `O(1)`.
- Cách 4, 5: `O(n)` thời gian, `O(n)` bộ nhớ phụ (mảng hậu tố).

Không có best/worst khác nhau (đều duyệt đúng một lần hoặc hai vòng cố định).

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (độ dài 1, toàn chẵn, toàn lẻ, xen kẽ) và 500 input ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionSuffix(), SolutionTotals(), SolutionSufOdd(), SolutionAccumulate(), Solution()]

fixed = [
    ([1, 2, 3, 4], [2, 1, 1, 0]),
    ([1], [0]),
    ([2], [0]),
    ([2, 4, 6, 8], [0, 0, 0, 0]),
    ([1, 3, 5], [0, 0, 0]),
    ([1, 2, 1, 2], [2, 1, 1, 0]),
    ([2, 1], [1, 0]),
    ([1, 2], [1, 0]),
    ([100, 99, 98], [1, 1, 0]),
]
for nums, expected in fixed:
    for s in solutions:
        assert s.countOppositeParity(list(nums)) == expected, (type(s).__name__, nums)

random.seed(39170)
for _ in range(500):
    n = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    outs = [s.countOppositeParity(list(nums)) for s in solutions]
    assert all(o == outs[0] for o in outs), (nums, outs)
    ans = outs[0]
    assert len(ans) == n and ans[-1] == 0 and all(0 <= a <= n - 1 - i for i, a in enumerate(ans))
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Tổng điểm bằng số **cặp** `(i, j)` với `i < j` khác parity, tức `E * O` (E chẵn, O lẻ). (2) Điểm không đổi nếu thay mỗi số bằng một số cùng parity bất kỳ. (3) Đảo parity toàn bộ (cộng 1) không đổi đáp án.

In [17]:
random.seed(3917)
for _ in range(300):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 100))]
    ans = Solution().countOppositeParity(nums)
    E = sum(1 for x in nums if x % 2 == 0)
    O = len(nums) - E
    assert sum(ans) == E * O
    same_parity = [x if random.random() < 0.5 else (x + 2 if x + 2 <= 100 else x - 2) for x in nums]
    assert Solution().countOppositeParity(same_parity) == ans
    assert Solution().countOppositeParity([x + 1 for x in nums]) == ans
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm cả các chỉ số `j < i`

Đề chỉ đếm `j > i`. Nếu đếm mọi `j != i`, kết quả sẽ lớn hơn.

In [18]:
def wrong_both_sides(nums):
    n = len(nums)
    return [sum(1 for j in range(n) if j != i and nums[i] % 2 != nums[j] % 2) for i in range(n)]    # SAI: cả j < i

print(wrong_both_sides([1, 2, 3, 4]), "!=", Solution().countOppositeParity([1, 2, 3, 4]))
assert wrong_both_sides([1, 2, 3, 4]) == [2, 2, 2, 2] and Solution().countOppositeParity([1, 2, 3, 4]) == [2, 1, 1, 0]

[2, 2, 2, 2] != [2, 1, 1, 0]


### Lỗi 2: đếm cùng parity thay vì khác parity

Lấy nhầm nhóm: nếu `nums[i]` lẻ ta phải đếm số **chẵn** phía sau, không phải số lẻ.

In [19]:
def wrong_same_parity(nums):
    n = len(nums)
    ans, even, odd = [0] * n, 0, 0
    for i in range(n - 1, -1, -1):
        ans[i] = odd if nums[i] % 2 == 1 else even                 # SAI: đảo nhóm
        if nums[i] % 2 == 0:
            even += 1
        else:
            odd += 1
    return ans

print(wrong_same_parity([1, 2, 3, 4]), "!=", Solution().countOppositeParity([1, 2, 3, 4]))
assert wrong_same_parity([1, 2, 3, 4]) == [1, 1, 0, 0] and Solution().countOppositeParity([1, 2, 3, 4]) == [2, 1, 1, 0]

[1, 1, 0, 0] != [2, 1, 1, 0]


### Lỗi 3: duyệt **xuôi** với bộ đếm tích lũy (đếm phía **trước**, không phải phía sau)

Giữ `cnt` khi duyệt từ trái sang phải cho ta số phần tử **trước** `i`. Đề cần phía **sau**.

In [20]:
def wrong_forward(nums):
    cnt = [0, 0]
    ans = []
    for x in nums:
        ans.append(cnt[1 - x % 2])                                   # SAI: cnt chỉ chứa phần tử phía trước
        cnt[x % 2] += 1
    return ans

print(wrong_forward([1, 2, 3, 4]), "!=", Solution().countOppositeParity([1, 2, 3, 4]))
assert wrong_forward([1, 2, 3, 4]) == [0, 1, 1, 2] and Solution().countOppositeParity([1, 2, 3, 4]) == [2, 1, 1, 0]

[0, 1, 1, 2] != [2, 1, 1, 0]


### Lỗi 4: dùng `nums[i] / 2` hoặc so sánh giá trị thay vì parity

`nums[i] != nums[j]` luôn đúng với hai số phân biệt nên đếm hết; parity chỉ là `% 2`.

In [21]:
def wrong_compare_values(nums):
    n = len(nums)
    return [sum(1 for j in range(i + 1, n) if nums[i] != nums[j]) for i in range(n)]       # SAI: so giá trị

print(wrong_compare_values([1, 2, 3, 4]), "!=", Solution().countOppositeParity([1, 2, 3, 4]))
assert wrong_compare_values([1, 2, 3, 4]) == [3, 2, 1, 0] and Solution().countOppositeParity([1, 2, 3, 4]) == [2, 1, 1, 0]

[3, 2, 1, 0] != [2, 1, 1, 0]


### Lỗi 5: trả về một số (tổng) thay vì mảng

Đề yêu cầu mảng `answer` độ dài `n`. Trả tổng điểm là sai kiểu.

In [22]:
def wrong_total(nums):
    return sum(Solution().countOppositeParity(nums))               # SAI: trả một số

print(wrong_total([1, 2, 3, 4]), "!=", Solution().countOppositeParity([1, 2, 3, 4]))
assert wrong_total([1, 2, 3, 4]) == 4 and isinstance(Solution().countOppositeParity([1, 2, 3, 4]), list)

4 != [2, 1, 1, 0]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số phần tử phía sau thỏa tính chất" -> **duyệt ngược** và giữ bộ đếm theo nhóm.
- Tính chất chỉ có vài giá trị (chẵn/lẻ) -> **bộ đếm theo nhóm** thay vì so từng cặp: `cnt[x % 2]`.
- Luôn **đọc trước, cập nhật sau** khi chỉ đếm phía sau (hoặc **bỏ trước, đọc sau** khi đếm từ tổng).
- Tổng của mọi điểm bằng số cặp khác parity `E × O`: dùng để tự kiểm tra.
- Với `n <= 100` brute force vẫn qua, nhưng luyện thói quen tối ưu `O(n)`.

## 15. Biến thể của bài

1. **Count of Smaller Numbers After Self** (315): đếm số nhỏ hơn phía sau (cần cấu trúc dữ liệu mạnh hơn).
2. **Number of Pairs With Opposite Parity**-style: đếm cặp thay vì từng chỉ số.
3. **Count Pairs of Equal Substrings / Good Pairs** (1512): đếm cặp bằng nhau bằng `Counter`.
4. **Product of Array Except Self** (238): tiền tố/hậu tố.
5. **Sort Array By Parity** (905): phân loại theo parity.
6. Biến thể: đếm phía **trước** (`j < i`), hoặc cả hai phía, hoặc theo modulo `k` bất kỳ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | mọi cặp `i < j` | `O(n²)` | `O(n)` | Oracle |
| 2. Hậu tố | duyệt ngược, `even`/`odd` | `O(n)` | `O(n)` kết quả | Nộp bài |
| 3. Tổng rồi bỏ dần | `cnt` toàn mảng, trừ dần | `O(n)` | `O(n)` kết quả | Duyệt từ trái |
| 4. Mảng số lẻ hậu tố | công thức `(n-1-i) - odd` | `O(n)` | `O(n)` | Khi cần mảng hậu tố |
| 5. `accumulate` | tổng tích lũy bit | `O(n)` | `O(n)` | Ngắn gọn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().countOppositeParity([2, 3, 4, 5])` và `Solution().countOppositeParity([7, 7, 7])` ra gì?

**Bài 2 (tính tay):** với `nums = [5, 2, 8, 3, 6]`, tính `answer` bằng phương pháp đếm hậu tố.

**Bài 3 (điền TODO):** hoàn thành `count_same_parity(nums)`: với mỗi `i`, đếm `j > i` **cùng** parity.

**Bài 4 (tự viết lại):** viết `count_opposite_before(nums)`: với mỗi `i`, đếm `j < i` khác parity.

**Bài 5 (mini challenge):** viết `count_opposite_mod(nums, k)`: đếm `j > i` với `nums[j] % k != nums[i] % k` (tổng quát hóa parity `k = 2`), dùng `Counter` các dư.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().countOppositeParity([2, 3, 4, 5]), Solution().countOppositeParity([7, 7, 7]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_same_parity_todo(nums):
    ans = [0] * len(nums)
    # TODO: duyệt ngược, dùng cnt[x % 2] của các phần tử phía sau
    return ans

[2, 1, 1, 0] [0, 0, 0]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[7,7,7]` toàn lẻ -> không có số chẵn. `[2,3,4,5]`: xem từng vị trí. Bài 2: duyệt từ `6` về `5`.

**Mức 2 (vừa):** Bài 3: đọc `cnt[x % 2]` (cùng nhóm). Bài 4: duyệt **xuôi** và đếm các phần tử phía trước. Bài 5: dùng `Counter` các dư `x % k` của hậu tố, điểm là `tổng - cnt[x % k]`.

**Mức 3 (gần đáp án):** Bài 1: `[2, 1, 1, 0]` (số `2` chẵn, phía sau `3, 4, 5` có hai số lẻ) và `[0, 0, 0]`. Bài 2: `[3, 1, 1, 1, 0]`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().countOppositeParity([2, 3, 4, 5]) == [2, 1, 1, 0] and Solution().countOppositeParity([7, 7, 7]) == [0, 0, 0]
assert Solution().countOppositeParity([5, 2, 8, 3, 6]) == [3, 1, 1, 1, 0]
assert Solution().countOppositeParity([5, 2, 8, 3, 6]) == SolutionBrute().countOppositeParity([5, 2, 8, 3, 6])

# Bài 3
def count_same_parity(nums):
    cnt = [0, 0]
    ans = [0] * len(nums)
    for i in range(len(nums) - 1, -1, -1):
        ans[i] = cnt[nums[i] % 2]
        cnt[nums[i] % 2] += 1
    return ans
assert count_same_parity([1, 2, 3, 4]) == [1, 1, 0, 0] and count_same_parity([7, 7, 7]) == [2, 1, 0]

# Bài 4
def count_opposite_before(nums):
    cnt = [0, 0]
    ans = []
    for x in nums:
        ans.append(cnt[1 - x % 2])
        cnt[x % 2] += 1
    return ans
assert count_opposite_before([1, 2, 3, 4]) == [0, 1, 1, 2] and count_opposite_before([2]) == [0]

# Bài 5
from collections import Counter
def count_opposite_mod(nums, k):
    cnt = Counter()
    ans = [0] * len(nums)
    total = 0
    for i in range(len(nums) - 1, -1, -1):
        ans[i] = total - cnt[nums[i] % k]
        cnt[nums[i] % k] += 1
        total += 1
    return ans
assert count_opposite_mod([1, 2, 3, 4], 2) == Solution().countOppositeParity([1, 2, 3, 4])
random.seed(17)
for _ in range(100):
    a = [random.randint(1, 100) for _ in range(random.randint(1, 30))]
    assert count_opposite_mod(a, 2) == Solution().countOppositeParity(a)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm phía sau theo nhóm":**

```python
cnt = [0, 0]                       # cnt[0]: số chẵn, cnt[1]: số lẻ (của phần tử phía sau)
ans = [0] * n
for i in range(n - 1, -1, -1):
    ans[i] = cnt[1 - nums[i] % 2]  # ĐỌC: đếm nhóm KHÁC parity
    cnt[nums[i] % 2] += 1          # CẬP NHẬT: thêm nums[i] sau khi đọc
return ans
```

**Checklist trước khi submit:**

- [ ] Chỉ đếm `j > i` (phía sau), không đếm `j < i` và không đếm chính `i`?
- [ ] Đếm nhóm **khác** parity (lẻ -> đếm chẵn, chẵn -> đếm lẻ)?
- [ ] Đọc bộ đếm **trước**, cập nhật **sau**?
- [ ] Trả về **mảng** độ dài `n`?
- [ ] Đã thử `[1]`, `[1,2,3,4]`, toàn chẵn, toàn lẻ?
- [ ] Chữ ký đúng `countOppositeParity(self, nums: List[int]) -> List[int]`?